In [ ]:
!pip install langchain langgraph langchain-community chromadb sqlite-utils


In [ ]:
from langchain_community.llms import Ollama  # Ollama LLM 사용
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿
from langgraph.graph import StateGraph, END  # LangGraph 상태 머신
from typing import TypedDict  # 타입 정의용

# 1. 상태 정의
class AgentState(TypedDict):  # 상태 타입 정의
    query: str  # 사용자 질의
    symptoms: str  # 추출된 증상
    disease_candidates: str  # 질병 후보
    result: str  # 최종 응답

# 2. LLM 초기화
llm = Ollama(model="exaone3.5:7.8b")  # Ollama 모델 로딩

# 3. 에이전트 정의
extractor_prompt = PromptTemplate.from_template("""
                                                사용자의 질문에서 증상에 해당하는 단어 또는 구를 추출하세요.  
                                                결과는 쉼표로 구분된 문자열로 출력하세요.  
                                                질문: {query}
                                                """)  # 증상 추출 프롬프트

def extractor_agent(state: AgentState):  # 증상 추출 함수
    chain = extractor_prompt | llm  # 프롬프트 체인
    symptoms = chain.invoke({"query": state["query"]})  # LLM 실행
    return {**state, "symptoms": symptoms.strip()}  # 상태에 추가

matcher_prompt = PromptTemplate.from_template("""
                                                다음 증상 목록을 바탕으로 가장 가능성 높은 질병 이름 3개를 쉼표로 추정하세요.
                                                증상: {symptoms}
                                                """)  # 질병 후보 추정 프롬프트

def matcher_agent(state: AgentState):  # 질병 후보 추정
    chain = matcher_prompt | llm
    candidates = chain.invoke({"symptoms": state["symptoms"]})
    return {**state, "disease_candidates": candidates.strip()}

answer_prompt = PromptTemplate.from_template("""
                                            사용자의 증상은 다음과 같습니다: {symptoms}

                                            예측된 질병 후보: {disease_candidates}

                                            위 내용을 바탕으로 사용자에게 알기 쉽게 설명해주세요.
                                            """)  # 최종 응답 생성 프롬프트

def answer_agent(state: AgentState):  # 응답 생성 에이전트
    chain = answer_prompt | llm  # 프롬프트와 LLM을 연결하여 실행 체인 구성
    answer = chain.invoke({
        "symptoms": state["symptoms"],
        "disease_candidates": state["disease_candidates"]
    })
    return {**state, "result": answer.strip()}

# 4. LangGraph 정의
from langgraph.graph import StateGraph  # LangGraph 구성 요소

graph = StateGraph(AgentState)  # 그래프 정의
graph.add_node("extractor", extractor_agent)  # 노드 추가
graph.add_node("matcher", matcher_agent)
graph.add_node("answer", answer_agent)

graph.set_entry_point("extractor")  # 시작 노드 설정
graph.add_edge("extractor", "matcher")  # 노드 간 연결 정의
graph.add_edge("matcher", "answer")
graph.add_edge("answer", END)  # 종료 노드 설정

app = graph.compile()  # 그래프 컴파일

print("============================== LangGraph 구조:")
app.get_graph().print_ascii()  # 구조 출력

# 5. 실행 예시
query = "기침이 심하고 목이 아프고 열이 납니다"  # 사용자 질문
result = app.invoke({"query": query})  # 실행

print("============================== 최종 응답:")
print(result["result"])  # 결과 출력


/tmp/ipykernel_942723/1617558422.py:14: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="exaone3.5:7.8b")  # Ollama 모델 로딩


============================== LangGraph 구조:
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| extractor |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | matcher |   
 +---------+   
      *        
      *        
      *        
  +--------+   
  | answer |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   
============================== 최종 응답:
안녕하세요. 제공해주신 증상 (기침, 목 아픔, 열)을 바탕으로 몇 가지 가능한 질병에 대해 간단히 설명해 드리겠습니다:

1. **감기**:
   - **특징**: 감기는 가장 흔하게 겪는 호흡기 질환 중 하나입니다. 주로 콧물, 코막힘, 기침, 목 아픔, 가벼운 발열을 동반합니다. 증상은 대체로 경미하고 며칠 내에 호전되는 경향이 있습니다.
   - **예후**: 대부분의 경우 자가 치유가 가능하며, 충분한 휴식과 수분 섭취가 도움이 됩니다.

2. **독감 (인플루엔자)**:
   - **특징**: 독감은 감기보다 더 심한 증상을 동반합니다. 높은 발열, 심한 기침, 근육통, 두통, 피로감 등이 특징입니다. 감기와 비슷한 증상이 있지만, 증상의 강도가 더 높을 수 있습니다.
   - **예후**: 빠른 치료와 휴식이 중요하며, 경우에 따라 항바이러스 약물이 처방될 수 있습니다.

3. **코로나19**:
   - **특징**: 코로나19는 기침, 목 